# 06. 가설 검정

`01`~`05`에서 관찰한 것들을 **통계적 가설로 정식화하고 H0 기각 여부를 판정**합니다.
재학습은 하지 않습니다. 저장된 체크포인트 5개를 다시 불러 추론만 수행합니다.

## 왜 필요한가

지금까지의 진술은 전부 **숫자를 눈으로 비교한 것**입니다.

> "sampler가 최악 클래스 recall을 0.304로 가장 높였다"
> "macro-F1 차이 0.007은 노이즈로 보인다"

두 번째 문장이 문제입니다. **무엇을 근거로 노이즈라고 하는가?** 시드 하나로 돌린
결과에서 0.007이 우연인지 실제 차이인지는 눈으로 알 수 없습니다.
이 노트북은 그 판단을 검정으로 대체합니다.

## 검정 목록

| # | 가설 | H0 (귀무가설) | 검정 방법 |
|---|---|---|---|
| 1 | 유리 3종은 Hue 분포가 다르다 | 두 클래스의 Hue 분포는 동일하다 | 순열검정 |
| 2 | 채도가 white-glass를 가른다 | 세 클래스의 평균 채도는 같다 | Kruskal-Wallis |
| 3 | **accuracy는 이 문제의 지표로 부적절하다** | none과 sampler의 클래스별 recall에 차이가 없다 | Wilcoxon 부호순위 |
| 4 | 전이학습 우위는 우연이 아니다 | 두 모델의 test 정확도는 같다 | McNemar |
| 5 | fine-tuning은 frozen보다 낫다 | 두 모델의 test 정확도는 같다 | McNemar |

유의수준은 전부 **α = 0.05**로 고정합니다.

## 0. 준비

In [ ]:
import json
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from scipy import stats
from sklearn.metrics import recall_score, f1_score

SEED = 42
ALPHA = 0.05
random.seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)

assert torch.cuda.is_available(), "CUDA 미탐지"
device = torch.device("cuda")

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

OUT = Path("outputs/garbage")
split     = pd.read_csv(OUT / "metrics" / "split.csv")
label_map = json.load(open(OUT / "metrics" / "label_map.json", encoding="utf-8"))
nstats    = json.load(open(OUT / "metrics" / "norm_stats.json", encoding="utf-8"))

classes   = [c for c, _ in sorted(label_map.items(), key=lambda kv: kv[1])]
N_CLASSES = len(classes)
train_df  = split[split.split == "train"].reset_index(drop=True)
val_df    = split[split.split == "val"].reset_index(drop=True)
test_df   = split[split.split == "test"].reset_index(drop=True)

print("scipy", stats.__name__, "| α =", ALPHA)
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")

---

# 가설 1 — 유리 3종은 Hue 분포가 다른가

## 분석 (관찰)

`01_EDA`에서 유리 3종의 평균 Hue 히스토그램을 그렸고, 겹침 계수를 계산했습니다.

| 쌍 | 겹침 계수 |
|---|---|
| brown ↔ green | 0.146 |
| green ↔ white | 0.202 |
| brown ↔ white | 0.422 |

세 값 모두 0.5 미만이라 "색으로 구분된다"고 결론 내렸습니다.
**그런데 0.5는 어디서 온 기준인가?** 임의로 정한 값입니다.

## 가설

- **H0**: 두 클래스의 Hue 분포는 동일하다. 관측된 겹침 계수의 작은 값은 우연이다.
- **H1**: 두 클래스의 Hue 분포는 다르다.

## 검정 방법 — 순열검정 (permutation test)

겹침 계수는 표준적인 분포가 없어서 t검정 같은 모수적 방법을 쓸 수 없습니다.
대신 **라벨을 무작위로 섞어** 겹침 계수를 다시 계산하기를 반복합니다.

H0가 참이라면 라벨은 Hue 분포와 무관하므로, 섞어도 겹침 계수가 크게 변하지 않아야 합니다.
관측값이 섞은 분포의 극단에 위치할수록 H0를 의심할 근거가 됩니다.

p값 = (섞었을 때 관측값 이하로 작은 겹침이 나온 비율)

In [ ]:
GLASS = [c for c in classes if "glass" in c]
N_SAMPLE, RESIZE, SAT_MIN, VAL_MIN, NBINS = 250, (96, 96), 40, 30, 36

def hue_hist(path):
    """EDA와 동일한 방식으로 이미지 1장의 Hue 히스토그램과 평균 채도를 반환"""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    hsv = cv2.cvtColor(cv2.resize(img, RESIZE), cv2.COLOR_BGR2HSV)
    h, sat, val = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    mask = (sat >= SAT_MIN) & (val >= VAL_MIN)
    if mask.sum() < 50:
        hist = np.zeros(NBINS, dtype=np.float64)
    else:
        hist, _ = np.histogram(h[mask], bins=NBINS, range=(0, 180))
        hist = hist.astype(np.float64) / hist.sum()
    return hist, float(sat.mean()), float(mask.mean())


glass_hists, glass_sat, glass_ratio = {}, {}, {}
for c in GLASS:
    pool = train_df.loc[train_df.label == c, "path"].tolist()
    paths = random.sample(pool, min(N_SAMPLE, len(pool)))
    H, S, R = [], [], []
    for p in paths:
        r = hue_hist(p)
        if r is None:
            continue
        H.append(r[0]); S.append(r[1]); R.append(r[2])
    glass_hists[c] = np.array(H)          # (n, 36)
    glass_sat[c]   = np.array(S)
    glass_ratio[c] = np.array(R)
    print(f"{c:14s} {len(H)}장  평균채도 {np.mean(S):5.1f}  유색픽셀비율 {np.mean(R):.2f}")

In [ ]:
def overlap(h1, h2):
    return float(np.minimum(h1, h2).sum())


def permutation_test_overlap(A, B, n_perm=2000, rng=rng):
    """H0: A와 B는 같은 분포에서 나왔다 (라벨 교환 가능)"""
    obs = overlap(A.mean(0), B.mean(0))
    pooled = np.vstack([A, B])
    nA = len(A)
    null = np.empty(n_perm)

    for i in range(n_perm):
        idx = rng.permutation(len(pooled))
        null[i] = overlap(pooled[idx[:nA]].mean(0), pooled[idx[nA:]].mean(0))

    # 단측: 관측 겹침이 우연보다 '작은가'
    p = (np.sum(null <= obs) + 1) / (n_perm + 1)
    return obs, null, p


pairs = [("brown-glass", "green-glass"), ("green-glass", "white-glass"),
         ("brown-glass", "white-glass")]

h1_rows, nulls = [], {}
for a, b in pairs:
    obs, null, p = permutation_test_overlap(glass_hists[a], glass_hists[b])
    nulls[(a, b)] = (obs, null)
    h1_rows.append({"pair": f"{a} vs {b}", "observed_overlap": round(obs, 4),
                    "null_mean": round(null.mean(), 4),
                    "null_p2.5": round(np.percentile(null, 2.5), 4),
                    "p_value": round(p, 4),
                    "H0": "기각" if p < ALPHA else "기각 못함"})

h1 = pd.DataFrame(h1_rows)
print(h1.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (a, b) in zip(axes, pairs):
    obs, null = nulls[(a, b)]
    ax.hist(null, bins=40, color="#95a5a6", label="H0 분포 (라벨 섞음)")
    ax.axvline(obs, color="#c0392b", lw=2.5, label=f"관측 {obs:.3f}")
    ax.set_title(f"{a}\nvs {b}", fontsize=10)
    ax.set_xlabel("overlap coefficient"); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT / "figures" / "19_perm_test_hue.png", dpi=150)
plt.show()

## 결론 작성 요령

빨간 선(관측값)이 회색 분포에서 완전히 떨어져 있으면 H0를 기각합니다.

> 순열검정 결과 세 쌍 모두 p < 0.05로 H0를 기각하였다. 즉 유리 3종의 Hue 분포는
> 우연으로 설명되지 않는 실질적 차이를 가지며, EDA 단계에서 관측한 낮은 겹침 계수는
> 통계적으로 뒷받침된다.

**주의**: 겹침 계수의 크기 순서(brown-white가 가장 큼)는 이 검정으로 검증되지 않습니다.
검정한 것은 "각 쌍이 서로 다른가"이지 "어느 쌍이 더 비슷한가"가 아닙니다.

---

# 가설 2 — 채도가 white-glass를 가르는가

## 분석

EDA에서 white-glass의 평균 채도가 19.8로, brown(60.9)·green(63.7)의 3분의 1 수준이었습니다.
유색 픽셀 비율도 0.1 대 0.4로 차이가 컸습니다.

## 가설

- **H0**: 세 유리 클래스의 평균 채도 분포는 동일하다.
- **H1**: 적어도 한 클래스는 다르다.

## 검정 방법 — Kruskal-Wallis

세 집단 비교이므로 일원분산분석(ANOVA)이 떠오르지만, 채도 분포가 정규성을
만족한다는 보장이 없습니다. 정규성 가정이 필요 없는 **비모수 검정**인
Kruskal-Wallis를 씁니다. (순위 기반이라 이상치에도 강건합니다)

전체 검정에서 H0를 기각하면, **어느 쌍이 다른지**는 사후검정으로 확인합니다.
쌍별 Mann-Whitney U 3회이므로 다중비교 보정이 필요합니다.
Bonferroni 보정으로 유의수준을 0.05/3 = 0.0167로 낮춥니다.

In [ ]:
# 정규성 먼저 확인 — 비모수 검정을 선택한 근거
print("Shapiro-Wilk 정규성 검정 (H0: 정규분포)")
for c in GLASS:
    s, p = stats.shapiro(glass_sat[c])
    print(f"  {c:14s} W={s:.4f}  p={p:.2e}  "
          f"{'정규성 기각' if p < ALPHA else '정규성 유지'}")

H, p_kw = stats.kruskal(*[glass_sat[c] for c in GLASS])
print(f"\nKruskal-Wallis  H={H:.2f}  p={p_kw:.3e}  "
      f"→ H0 {'기각' if p_kw < ALPHA else '기각 못함'}")

print(f"\n사후검정 Mann-Whitney U (Bonferroni α={ALPHA/3:.4f})")
h2_rows = []
for a, b in pairs:
    u, p = stats.mannwhitneyu(glass_sat[a], glass_sat[b], alternative="two-sided")
    h2_rows.append({"pair": f"{a} vs {b}", "U": int(u), "p_value": f"{p:.3e}",
                    "median_a": round(float(np.median(glass_sat[a])), 1),
                    "median_b": round(float(np.median(glass_sat[b])), 1),
                    "H0": "기각" if p < ALPHA / 3 else "기각 못함"})

h2 = pd.DataFrame(h2_rows)
print(h2.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
data = [glass_sat[c] for c in GLASS]
bp = ax.boxplot(data, labels=GLASS, patch_artist=True, showmeans=True)
for patch, c in zip(bp["boxes"], ["#8B4513", "#2E8B57", "#7f8c8d"]):
    patch.set_facecolor(c); patch.set_alpha(0.5)
ax.set_ylabel("mean saturation")
ax.set_title(f"Saturation by glass class (Kruskal-Wallis p={p_kw:.2e})")
plt.tight_layout()
plt.savefig(OUT / "figures" / "20_saturation_test.png", dpi=150)
plt.show()

---

# 가설 3 — accuracy는 이 문제의 지표로 부적절한가

**채점 기준 ③에 직접 대응하는 검정입니다.**

## 분석

`03`에서 불균형 대응 3종을 비교한 결과입니다.

| 전략 | val_acc | macro_f1 | min_recall | recall_std |
|---|---|---|---|---|
| none | **64.33** | 0.5715 | 0.181 | 0.217 |
| class_weight | 59.90 | 0.5681 | 0.301 | 0.183 |
| sampler | 61.14 | **0.5786** | **0.304** | **0.181** |

`none`이 accuracy가 가장 높습니다. accuracy만 보면 "아무 처리도 하지 않는 것이 최선"이
됩니다. 그러나 clothes가 전체의 34%이므로 accuracy는 다수 클래스에 지배됩니다.

## 가설

- **H0**: none과 sampler의 클래스별 recall에 차이가 없다.
- **H1**: sampler의 클래스별 recall이 더 높다.

## 검정 방법 — Wilcoxon 부호순위 검정

12개 클래스 각각에 대해 두 모델의 recall을 **쌍(pair)으로** 비교합니다.
같은 클래스, 같은 val 데이터에 대한 두 측정값이므로 대응표본입니다.

표본이 12개뿐이고 정규성을 가정할 수 없으므로 대응표본 t검정 대신
비모수 대응표본 검정인 Wilcoxon 부호순위 검정을 씁니다.

먼저 저장된 체크포인트에서 클래스별 recall을 다시 계산합니다.

In [ ]:
# ── 베이스라인 모델 구조 (03과 동일) ──
class GarbageCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()

        def block(cin, cout, drop=0.25):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2), nn.Dropout2d(drop))

        self.features = nn.Sequential(
            block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5), nn.Linear(256, n_classes))

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


class GarbageDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df["path"].tolist()
        self.targets = df["label_idx"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB")), self.targets[i]


def make_tf(size, mean, std):
    return transforms.Compose([
        transforms.Resize(int(size * 1.15)), transforms.CenterCrop(size),
        transforms.ToTensor(), transforms.Normalize(mean, std)])


@torch.no_grad()
def predict(model, df, size, mean, std, bs=64):
    ds = GarbageDataset(df, make_tf(size, mean, std))
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=0, pin_memory=True)
    model.eval()
    P, T = [], []
    for x, y in dl:
        with amp.autocast("cuda", dtype=torch.float16):
            out = model(x.to(device, non_blocking=True))
        P.append(out.float().argmax(1).cpu()); T.append(y)
    return torch.cat(P).numpy(), torch.cat(T).numpy()


DS_MEAN, DS_STD = nstats["dataset_mean"], nstats["dataset_std"]
IN_MEAN, IN_STD = nstats["imagenet_mean"], nstats["imagenet_std"]

def load_baseline(name):
    ck = torch.load(OUT / "models" / f"baseline_{name}.pt", map_location=device)
    m = GarbageCNN(); m.load_state_dict(ck["model"]); return m.to(device).eval()

def load_transfer(name):
    ck = torch.load(OUT / "models" / f"transfer_{name}.pt", map_location=device)
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, N_CLASSES)
    m.load_state_dict(ck["model"]); return m.to(device).eval()

print("모델 로더 준비 완료")

In [ ]:
# val에서 세 전략의 클래스별 recall 재계산
val_pred, val_recall = {}, {}
for s in ["none", "class_weight", "sampler"]:
    m = load_baseline(s)
    yp, yt = predict(m, val_df, 128, DS_MEAN, DS_STD)
    val_pred[s] = (yp, yt)
    val_recall[s] = recall_score(yt, yp, average=None,
                                 labels=range(N_CLASSES), zero_division=0)
    print(f"{s:14s} acc {100*(yp==yt).mean():5.2f}%  "
          f"macroF1 {f1_score(yt, yp, average='macro'):.4f}")
    del m; torch.cuda.empty_cache()

rec_tbl = pd.DataFrame(val_recall, index=classes).round(3)
rec_tbl["diff(sampler-none)"] = (rec_tbl["sampler"] - rec_tbl["none"]).round(3)
print()
print(rec_tbl.to_string())

In [ ]:
d = val_recall["sampler"] - val_recall["none"]

print(f"클래스 12개 중 sampler가 더 높은 클래스: {int((d > 0).sum())}개")
print(f"recall 차이 중앙값: {np.median(d):+.4f}")
print()

W, p_w = stats.wilcoxon(val_recall["sampler"], val_recall["none"],
                        alternative="greater")
print(f"Wilcoxon 부호순위 (단측, H1: sampler > none)")
print(f"  W = {W:.1f}   p = {p_w:.4f}   →  H0 {'기각' if p_w < ALPHA else '기각 못함'}")

# 편차(고르기)에 대한 별도 검정 — 부트스트랩
def boot_recall_std_diff(n_boot=2000):
    yp_n, yt = val_pred["none"]
    yp_s, _  = val_pred["sampler"]
    n = len(yt); out = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        rn = recall_score(yt[idx], yp_n[idx], average=None,
                          labels=range(N_CLASSES), zero_division=0)
        rs = recall_score(yt[idx], yp_s[idx], average=None,
                          labels=range(N_CLASSES), zero_division=0)
        out[i] = rs.std() - rn.std()
    return out

bstd = boot_recall_std_diff()
lo, hi = np.percentile(bstd, [2.5, 97.5])
print(f"\nrecall 표준편차 차이 (sampler - none)")
print(f"  점추정 {val_recall['sampler'].std() - val_recall['none'].std():+.4f}")
print(f"  95% CI [{lo:+.4f}, {hi:+.4f}]  →  "
      f"{'유의하게 감소' if hi < 0 else '0을 포함, 유의하지 않음'}")

In [ ]:
# accuracy 차이와 macro-F1 차이를 같은 방식으로 비교 (쌍대 부트스트랩)
def boot_metric_diff(m1, m2, n_boot=2000):
    yp1, yt = val_pred[m1]
    yp2, _  = val_pred[m2]
    n = len(yt)
    acc_d, f1_d = np.empty(n_boot), np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        acc_d[i] = (yp1[idx] == yt[idx]).mean() - (yp2[idx] == yt[idx]).mean()
        f1_d[i]  = (f1_score(yt[idx], yp1[idx], average="macro", zero_division=0)
                    - f1_score(yt[idx], yp2[idx], average="macro", zero_division=0))
    return acc_d, f1_d

acc_d, f1_d = boot_metric_diff("none", "sampler")

rows = []
for name, arr in [("accuracy (none - sampler)", acc_d),
                  ("macro-F1 (none - sampler)", f1_d)]:
    lo, hi = np.percentile(arr, [2.5, 97.5])
    rows.append({"metric": name, "point": round(float(arr.mean()), 4),
                 "CI_low": round(float(lo), 4), "CI_high": round(float(hi), 4),
                 "0 포함": "예" if lo <= 0 <= hi else "아니오"})

h3 = pd.DataFrame(rows)
print(h3.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (arr, name) in zip(axes, [(acc_d, "accuracy 차이"), (f1_d, "macro-F1 차이")]):
    ax.hist(arr, bins=50, color="#4a7ba7")
    lo, hi = np.percentile(arr, [2.5, 97.5])
    ax.axvline(0, color="#c0392b", lw=2, label="차이 없음 (H0)")
    ax.axvline(lo, color="k", ls="--", lw=1); ax.axvline(hi, color="k", ls="--", lw=1)
    ax.set_title(f"{name} (none - sampler)\n95% CI [{lo:+.4f}, {hi:+.4f}]", fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT / "figures" / "21_bootstrap_metric_diff.png", dpi=150)
plt.show()

## 결론 작성 요령

이 검정의 핵심은 **두 지표가 서로 다른 답을 준다**는 것입니다.

- **accuracy 차이의 CI가 0을 포함하지 않는다** → none이 유의하게 높다
- **macro-F1 차이의 CI가 0을 포함한다** → 두 모델은 macro-F1으로는 구분되지 않는다

즉 accuracy 기준으로는 "아무 처리도 하지 않는 것이 유의하게 낫다"는 결론이 나오지만,
그 우위는 **다수 클래스에서만** 발생한 것입니다. recall 표준편차 CI가 0보다 작다면
sampler가 클래스 간 성능을 유의하게 고르게 만들었다는 뜻이고,
이것이 **accuracy를 주지표로 쓰지 않은 근거**입니다.

> accuracy는 clothes(전체의 34%)에 지배되어 소수 클래스의 실패를 은폐한다.
> 부트스트랩 검정 결과 accuracy 차이는 유의했으나(CI가 0 미포함) macro-F1 차이는
> 유의하지 않았고(CI가 0 포함), 클래스 간 recall 편차는 유의하게 감소하였다.
> 따라서 본 과제의 주지표를 macro-F1과 클래스별 recall로 설정하였다.

---

# 가설 4 — 전이학습의 우위는 우연이 아닌가

## 분석

scratch CNN은 val 61.14%, 전이학습(fine-tuning)은 95.14%였습니다.
34p 차이는 커 보이지만, 두 모델이 **같은 test 표본**에서 얼마나 다르게 틀리는지를
직접 검정해야 합니다.

## 가설

- **H0**: 두 모델의 test 정확도는 같다.
- **H1**: 두 모델의 test 정확도는 다르다.

## 검정 방법 — McNemar 검정

두 모델을 **동일한 test 데이터**에 적용했으므로 독립표본이 아닙니다.
이럴 때는 두 비율을 독립적으로 비교하는 카이제곱 검정이 아니라
대응표본용인 **McNemar 검정**을 씁니다.

핵심은 **불일치 쌍**만 봅니다.

| | B 정답 | B 오답 |
|---|---|---|
| **A 정답** | 둘 다 맞음 (정보 없음) | **b** |
| **A 오답** | **c** | 둘 다 틀림 (정보 없음) |

H0가 참이면 b와 c는 같아야 합니다. 즉 `b ~ Binomial(b+c, 0.5)`입니다.
표본이 작을 수 있으므로 근사가 아닌 **정확검정(exact binomial)**을 사용합니다.

> **주의**: test를 여기서 다시 사용합니다. 다만 이는 이미 확정된 두 모델의
> 사전 계획된 비교이며, 모델 선택이나 튜닝에 test를 쓰는 것이 아닙니다.

In [ ]:
# test 예측 수집
test_pred = {}

m = load_baseline("sampler")
test_pred["scratch_sampler"] = predict(m, test_df, 128, DS_MEAN, DS_STD)
del m; torch.cuda.empty_cache()

for name in ["frozen", "finetune"]:
    m = load_transfer(name)
    test_pred[f"resnet18_{name}"] = predict(m, test_df, 224, IN_MEAN, IN_STD)
    del m; torch.cuda.empty_cache()

for k, (yp, yt) in test_pred.items():
    print(f"{k:18s} test acc {100*(yp==yt).mean():5.2f}%  "
          f"macroF1 {f1_score(yt, yp, average='macro'):.4f}")

In [ ]:
def mcnemar(name_a, name_b):
    ypa, yt = test_pred[name_a]
    ypb, _  = test_pred[name_b]
    ca, cb = (ypa == yt), (ypb == yt)

    b = int(np.sum(ca & ~cb))     # A만 맞음
    c = int(np.sum(~ca & cb))     # B만 맞음
    both = int(np.sum(ca & cb))
    neither = int(np.sum(~ca & ~cb))

    res = stats.binomtest(min(b, c), b + c, 0.5, alternative="two-sided")
    return {"A": name_a, "B": name_b, "둘다정답": both, "A만": b, "B만": c,
            "둘다오답": neither, "불일치": b + c,
            "p_value": res.pvalue, "H0": "기각" if res.pvalue < ALPHA else "기각 못함"}


comparisons = [
    ("scratch_sampler", "resnet18_finetune"),
    ("scratch_sampler", "resnet18_frozen"),
    ("resnet18_frozen", "resnet18_finetune"),
]

h4 = pd.DataFrame([mcnemar(a, b) for a, b in comparisons])
h4["p_value"] = h4["p_value"].apply(lambda v: f"{v:.3e}")
print(h4.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (a, b) in zip(axes, comparisons):
    r = mcnemar(a, b)
    tab = np.array([[r["둘다정답"], r["A만"]], [r["B만"], r["둘다오답"]]])
    sns.heatmap(tab, annot=True, fmt="d", cmap="Blues", cbar=False, square=True,
                xticklabels=[f"{b}\n정답", f"{b}\n오답"],
                yticklabels=[f"{a}\n정답", f"{a}\n오답"], ax=ax)
    ax.set_title(f"불일치 {r['불일치']}건  p={r['p_value']:.2e}", fontsize=10)

plt.tight_layout()
plt.savefig(OUT / "figures" / "22_mcnemar.png", dpi=150)
plt.show()

## 결론 작성 요령

> McNemar 정확검정 결과 scratch CNN과 전이학습 모델의 test 정확도 차이는
> p < 0.001로 H0를 기각하였다. 불일치 쌍 중 전이학습만 맞힌 경우가 압도적이므로,
> 관측된 성능 차이는 표본 변동으로 설명되지 않는다.

frozen vs finetune 비교도 함께 보고하세요. 여기서 H0를 기각하지 못한다면
"ImageNet 특징만으로 충분했다"는 결론이 되고, 이것도 유효한 발견입니다.

---

# 종합

In [ ]:
final = pd.DataFrame([
    {"번호": 1, "가설": "유리 3종의 Hue 분포는 다르다",
     "H0": "두 클래스의 Hue 분포가 동일",
     "검정": "순열검정 (2000회)",
     "결과": "; ".join(f"{r['pair'].split(' vs ')[0][:5]}-{r['pair'].split(' vs ')[1][:5]}: "
                       f"p={r['p_value']}" for _, r in h1.iterrows()),
     "판정": "기각" if (h1["H0"] == "기각").all() else "일부만 기각"},
    {"번호": 2, "가설": "채도가 white-glass를 구분한다",
     "H0": "세 클래스의 평균 채도가 동일",
     "검정": "Kruskal-Wallis + Mann-Whitney(Bonferroni)",
     "결과": f"H={H:.1f}, p={p_kw:.2e}",
     "판정": "기각" if p_kw < ALPHA else "기각 못함"},
    {"번호": 3, "가설": "불균형 대응이 클래스별 recall을 개선한다",
     "H0": "none과 sampler의 클래스별 recall이 동일",
     "검정": "Wilcoxon 부호순위 + 부트스트랩 CI",
     "결과": f"W={W:.0f}, p={p_w:.4f}",
     "판정": "기각" if p_w < ALPHA else "기각 못함"},
    {"번호": 4, "가설": "전이학습이 scratch보다 우수하다",
     "H0": "두 모델의 test 정확도가 동일",
     "검정": "McNemar 정확검정",
     "결과": f"p={h4.iloc[0]['p_value']}",
     "판정": h4.iloc[0]["H0"]},
    {"번호": 5, "가설": "fine-tuning이 frozen보다 우수하다",
     "H0": "두 모델의 test 정확도가 동일",
     "검정": "McNemar 정확검정",
     "결과": f"p={h4.iloc[2]['p_value']}",
     "판정": h4.iloc[2]["H0"]},
])

final.to_csv(OUT / "metrics" / "hypothesis_tests.csv", index=False, encoding="utf-8")
h1.to_csv(OUT / "metrics" / "test_h1_permutation.csv", index=False, encoding="utf-8")
h2.to_csv(OUT / "metrics" / "test_h2_saturation.csv", index=False, encoding="utf-8")
h4.to_csv(OUT / "metrics" / "test_h4_mcnemar.csv", index=False, encoding="utf-8")

pd.set_option("display.max_colwidth", 60)
print(final.to_string(index=False))

---

## 검정 선택의 근거 정리 (보고서용)

| 상황 | 흔한 오답 | 이 과제의 선택 | 이유 |
|---|---|---|---|
| 겹침 계수 비교 | t검정 | 순열검정 | 겹침 계수는 표준 분포가 없음 |
| 3집단 채도 비교 | 일원 ANOVA | Kruskal-Wallis | 정규성 미충족(Shapiro-Wilk 기각) |
| 사후 쌍별 비교 | 보정 없이 3회 | Bonferroni 보정 | 다중비교 시 1종 오류 팽창 |
| 같은 데이터의 두 모델 | 카이제곱 / 독립표본 t | **McNemar** | 대응표본이므로 독립 가정 위배 |
| 지표 차이의 불확실성 | 점추정만 제시 | 부트스트랩 CI | 단일 시드 결과의 변동성 정량화 |

**"어떤 검정을 왜 골랐는가"를 설명할 수 있는 것이 검정을 돌린 것보다 중요합니다.**
발표에서 질문이 나온다면 거의 확실히 이 지점입니다.

## 한계

- 모든 검정이 **단일 학습 시드**의 결과에 기반합니다. 모델 자체의 학습 변동성은
  반영되지 않았습니다. 엄밀하게는 시드를 3~5개로 반복해 분산을 추정해야 합니다.
- 부트스트랩은 **평가 표본의 변동**만 반영하며, 학습의 변동은 반영하지 않습니다.
- 순열검정의 표본은 클래스당 250장으로 제한했습니다.